# 5장. 분석을 믿을 수 있게 만드는 데이터 전처리

이 노트북은 `book/chapters/ch05_data_preprocessing.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 데이터를 무조건 깨끗하게 만드는 것이 아니라, 분석 결과를 신뢰할 수 있도록 데이터의 상태를 확인하고 처리 기준을 남기는 것입니다.


## 0. 제출 정보

- 이름: 김용준
- GitHub ID: rotcev40
- 작성일: 2026-10-14
- 최종 제출 URL: https://github.com/rotcev40/llm-data-analysis-study/blob/main/chapter05/chapter05.ipynb

## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 원본 데이터는 `data/raw/`에 그대로 둡니다.
- 전처리 결과는 `data/processed/`에 별도로 저장합니다.
- 전처리 요약 보고서는 `reports/ch05_preprocessing_summary.md`에 저장합니다.
- 코드가 실행되었다고 끝이 아니라, 처리 기준을 설명할 수 있어야 합니다.


## 1. 왜 전처리가 필요한가

현실의 데이터는 처음부터 분석하기 좋은 형태로 주어지지 않습니다.

예를 들어 다음과 같은 문제가 있을 수 있습니다.

| 문제 유형 | 예시 | 확인할 질문 |
|---|---|---|
| 결측치 | 나이가 비어 있는 고객 | 비어 있는 이유가 무엇인가? 삭제해도 되는가? |
| 중복 | 같은 고객 ID가 여러 번 등장 | 자연스러운 중복인가, 데이터 오류인가? |
| 타입 오류 | 가격이 문자열로 저장됨 | 계산 가능한 숫자형으로 바꿀 수 있는가? |
| 날짜 오류 | 주문일이 문자열로 저장됨 | 월별·요일별 분석에 사용할 수 있는가? |
| 문자열 표기 차이 | `Seoul`, ` Seoul`, `SEOUL` | 같은 값을 하나의 표기로 통일해야 하는가? |
| 이상값 | 수량이 음수이거나 가격이 0원 | 실제 의미가 있는 값인가, 입력 오류인가? |
| 파생 컬럼 필요 | 수량과 단가만 있고 주문금액이 없음 | 분석에 필요한 새 컬럼을 만들 수 있는가? |


## 2. 원본 데이터와 전처리 데이터 분리하기

전처리에서 가장 중요한 원칙은 원본 데이터를 직접 덮어쓰지 않는 것입니다.

```text
data/
├─ raw/
│  ├─ customers.csv
│  ├─ products.csv
│  ├─ orders.csv
│  └─ order_items.csv
└─ processed/
   ├─ customers_clean.csv
   ├─ products_clean.csv
   ├─ orders_clean.csv
   └─ order_items_clean.csv
```

원본과 전처리 결과를 분리하면 실수했을 때 원본으로 돌아갈 수 있고, 전처리 전후 차이를 비교할 수 있습니다.


## 3. 패키지와 경로 설정

노트북이 `notebooks/` 폴더 안에서 실행되는 경우와 프로젝트 루트에서 실행되는 경우를 모두 고려해 경로를 설정합니다.


In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == 'notebooks':
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

# Chapter 05는 공통 data/raw가 아니라 전용 raw 데이터를 사용한다
RAW_DIR = PROJECT_ROOT / 'practice' / 'chapter05' / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('원본 데이터 폴더:', RAW_DIR)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


현재 실행 위치: C:\Users\yjun0\Desktop\서울대\4학년\2학기\LLM을 이용한 빅데이터 분석\llm-data-analysis-course\notebooks
프로젝트 루트: C:\Users\yjun0\Desktop\서울대\4학년\2학기\LLM을 이용한 빅데이터 분석\llm-data-analysis-course
원본 데이터 폴더: C:\Users\yjun0\Desktop\서울대\4학년\2학기\LLM을 이용한 빅데이터 분석\llm-data-analysis-course\practice\chapter05\data\raw
전처리 데이터 폴더: C:\Users\yjun0\Desktop\서울대\4학년\2학기\LLM을 이용한 빅데이터 분석\llm-data-analysis-course\data\processed
보고서 폴더: C:\Users\yjun0\Desktop\서울대\4학년\2학기\LLM을 이용한 빅데이터 분석\llm-data-analysis-course\reports


## 4. 원본 데이터 불러오기

이번 장에서는 온라인 쇼핑몰 예제 데이터 4개를 사용합니다.

- `customers.csv`: 고객 정보
- `products.csv`: 상품 정보
- `orders.csv`: 주문 정보
- `order_items.csv`: 주문 상세 정보


In [2]:
customers = pd.read_csv(RAW_DIR / 'customers.csv')
products = pd.read_csv(RAW_DIR / 'products.csv')
orders = pd.read_csv(RAW_DIR / 'orders.csv')
order_items = pd.read_csv(RAW_DIR / 'order_items.csv')

print('원본 데이터 불러오기 완료')


원본 데이터 불러오기 완료


## 5. 전처리 전 데이터 크기 기록하기

전처리 전 데이터 크기를 기록해 두면 나중에 어떤 데이터가 얼마나 바뀌었는지 비교할 수 있습니다.


In [3]:
raw_data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

raw_shapes = pd.DataFrame({
    'dataset': list(raw_data.keys()),
    'rows': [df.shape[0] for df in raw_data.values()],
    'columns': [df.shape[1] for df in raw_data.values()],
})

raw_shapes


,dataset,rows,columns
0,customers,152,6
1,products,102,4
2,orders,302,5
3,order_items,765,5


In [4]:
for name, df in raw_data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()


[customers]
shape: (152, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (102, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (302, 5)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']

[order_items]
shape: (765, 5)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']



## 6. 원본을 복사해서 전처리 시작하기

전처리 과정에서는 원본 DataFrame을 직접 수정하지 않고 복사본을 사용합니다.


In [5]:
customers_clean = customers.copy()
products_clean = products.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()

clean_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

print('전처리용 복사본 생성 완료')


전처리용 복사본 생성 완료


## 7. 결측치 확인하기

결측치를 처리하기 전에 먼저 어디에 얼마나 비어 있는 값이 있는지 확인합니다. 개수와 비율을 함께 보면 데이터 크기에 따른 차이를 이해하기 쉽습니다.


In [6]:
for name, df in clean_data.items():
    print(f'\n[{name}] 결측치 개수')
    print(df.isna().sum())



[customers] 결측치 개수
customer_id    0
name           0
gender         0
age            4
city           2
signup_date    0
dtype: int64

[products] 결측치 개수
product_id      0
product_name    0
category        0
price           0
dtype: int64

[orders] 결측치 개수
order_id          0
customer_id       0
order_date        0
payment_method    0
order_status      0
dtype: int64

[order_items] 결측치 개수
order_item_id    0
order_id         0
product_id       0
quantity         0
unit_price       0
dtype: int64


In [7]:
def missing_summary(df):
    summary = pd.DataFrame({
        'missing_count': df.isna().sum(),
        'missing_ratio': (df.isna().mean() * 100).round(2),
    })
    return summary.sort_values('missing_count', ascending=False)

missing_summary(customers_clean)


,missing_count,missing_ratio
age,4,2.63
city,2,1.32
name,0,0.00
customer_id,0,0.00
gender,0,0.00
signup_date,0,0.00


## 8. 결측치 처리하기

결측치 처리는 컬럼의 의미에 따라 달라집니다.

- 나이처럼 숫자형 컬럼은 중앙값으로 대체할 수 있습니다.
- 도시처럼 범주형 컬럼은 `Unknown`으로 남겨 분석에서 구분할 수 있습니다.

실제 업무에서는 결측치가 왜 생겼는지 먼저 확인해야 합니다.


In [8]:
if 'age' in customers_clean.columns:
    customers_clean['age'] = pd.to_numeric(customers_clean['age'], errors='coerce')
    age_median = customers_clean['age'].median()
    customers_clean['age'] = customers_clean['age'].fillna(age_median)
    print('age 중앙값:', age_median)

if 'city' in customers_clean.columns:
    customers_clean['city'] = customers_clean['city'].fillna('Unknown')

missing_summary(customers_clean)


age 중앙값: 41.0


,missing_count,missing_ratio
customer_id,0,0.0
name,0,0.0
gender,0,0.0
age,0,0.0
city,0,0.0
signup_date,0,0.0


## 9. 중복 확인하기

중복은 맥락에 따라 의미가 다릅니다. `customers`의 `customer_id` 중복은 오류일 가능성이 크지만, `order_items`의 `order_id` 반복은 한 주문에 여러 상품이 들어갈 수 있으므로 자연스럽습니다.


In [9]:
for name, df in clean_data.items():
    print(name, '전체 행 중복 수:', df.duplicated().sum())


customers 전체 행 중복 수: 2
products 전체 행 중복 수: 2
orders 전체 행 중복 수: 2
order_items 전체 행 중복 수: 2


In [10]:
key_checks = {
    'customers': ('customer_id', customers_clean),
    'products': ('product_id', products_clean),
    'orders': ('order_id', orders_clean),
    'order_items': ('order_item_id', order_items_clean),
}

for name, (key_col, df) in key_checks.items():
    if key_col in df.columns:
        print(name, key_col, '중복 수:', df[key_col].duplicated().sum())
    else:
        print(name, key_col, '컬럼 없음')


customers customer_id 중복 수: 2
products product_id 중복 수: 2
orders order_id 중복 수: 2
order_items order_item_id 중복 수: 2


## 과제 1. 전처리 전 상태 기록

- 각 데이터 shape: customers (152, 6) / products (102, 4) / orders (302, 5) / order_items (765, 5)
- 결측치: customers의 age 4건(2.63%), city 2건(1.32%). 나머지 세 파일은 0건
- 전체 중복: 네 파일 모두 2건씩
- 주요 ID 중복: customer_id / product_id / order_id / order_item_id 모두 2건씩
- 타입 문제 후보: age, price, quantity, unit_price, order_date, signup_date가 전부 문자열

![전처리 전 상태](images/step01_before_1.png)
![전처리 전 상태](images/step01_before_2.png)
![전처리 전 상태](images/step01_before_3.png)
![전처리 전 상태](images/step01_before_4.png)

### 결과 관찰

read_csv 직후 dtype을 보면 ID 컬럼만 int64이고 나머지는 전부 str이다. 숫자여야 할 price, quantity, unit_price와 날짜여야 할 order_date, signup_date가 모두 문자열이다. isna로 잡힌 결측은 customers에만 6건 있다. 중복은 전체 행 기준과 ID 기준이 네 파일 모두 2건으로 같은 수가 나왔다.

### 나의 해석과 판단

타입 변환을 먼저 해야 한다. 지금 isna로 센 결측 6건은 칸이 비어 있는 경우만 센 것이고, age의 'unknown'처럼 문자열로 들어온 값은 결측으로 잡히지 않는다. 타입을 바꾸기 전에는 결측 개수도 이상값 범위도 제대로 셀 수 없다. 중복은 전체 행 중복과 ID 중복 수가 같으니 완전히 같은 행이 한 번씩 더 들어온 것으로 보이고, 이건 타입과 무관하게 먼저 지워도 괜찮을 것 같다.

### 업무·분석적 의미

매출 집계는 전부 price와 quantity를 곱하는 데서 시작한다. 이 두 컬럼이 문자열이면 합계가 아예 안 나오거나 문자열이 이어붙는 식으로 조용히 틀린 값이 나온다. 집계를 돌리기 전에 타입부터 확인해야 하는 이유다.

### 한계와 추가 확인 사항

ID 중복 2건이 완전히 같은 행인지, 같은 ID에 내용이 다른 행인지는 아직 구분하지 않았다. 후자라면 어느 쪽이 맞는지 원본 시스템에 물어봐야 한다. 그리고 지금 센 결측은 빈 칸만이라 실제 결측은 더 많을 것이다.

In [11]:
customers_clean = customers_clean.drop_duplicates()
products_clean = products_clean.drop_duplicates()
orders_clean = orders_clean.drop_duplicates()
order_items_clean = order_items_clean.drop_duplicates()

print('완전 중복 행 제거 완료')


완전 중복 행 제거 완료


## 10. 문자열 표기 정리하기

문자열 데이터에는 앞뒤 공백이 숨어 있을 수 있습니다. `Seoul`과 ` Seoul `은 눈으로는 비슷하지만 pandas에서는 서로 다른 값입니다.

결측치를 문자열 `nan`으로 바꾸지 않도록 주의하면서 문자열 컬럼의 앞뒤 공백을 제거합니다.


In [12]:
def strip_string_columns(df):
    result = df.copy()
    string_columns = result.select_dtypes(include='object').columns

    for col in string_columns:
        result[col] = result[col].where(
            result[col].isna(),
            result[col].astype(str).str.strip()
        )

    return result

customers_clean = strip_string_columns(customers_clean)
products_clean = strip_string_columns(products_clean)
orders_clean = strip_string_columns(orders_clean)
order_items_clean = strip_string_columns(order_items_clean)

print('문자열 앞뒤 공백 제거 완료')


문자열 앞뒤 공백 제거 완료


C:\Users\yjun0\AppData\Local\Temp\ipykernel_15244\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = result.select_dtypes(include='object').columns
C:\Users\yjun0\AppData\Local\Temp\ipykernel_15244\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/us

In [13]:
if 'city' in customers_clean.columns:
    display(customers_clean['city'].value_counts().head(10))

if 'order_status' in orders_clean.columns:
    display(orders_clean['order_status'].value_counts())


city
부산    23
대구    21
광주    21
서울    18
울산    17
인천    16
수원    10
성남     9
고양     8
대전     5
Name: count, dtype: int64

order_status
completed    175
cancelled     58
refunded      52
Complete       2
cancel         2
Refund         2
complete       1
COMPLETED      1
완료             1
Cancel         1
CANCELLED      1
취소             1
refund         1
REFUNDED       1
환불             1
Name: count, dtype: int64

## 11. 주문 상태값 통일하기

주문 상태값이 `complete`, `Complete`, `COMPLETED`, `완료`처럼 섞여 있으면 같은 상태를 서로 다른 값으로 집계할 수 있습니다. 대표 표기로 통일합니다.


In [14]:
status_map = {
    'complete': 'completed',
    'Complete': 'completed',
    'COMPLETED': 'completed',
    '완료': 'completed',
    'cancel': 'cancelled',
    'Cancel': 'cancelled',
    'CANCELLED': 'cancelled',
    '취소': 'cancelled',
    'refund': 'refunded',
    'Refund': 'refunded',
    'REFUNDED': 'refunded',
    '환불': 'refunded',
}

if 'order_status' in orders_clean.columns:
    orders_clean['order_status'] = orders_clean['order_status'].replace(status_map)
    display(orders_clean['order_status'].value_counts())


order_status
completed    180
cancelled     63
refunded      57
Name: count, dtype: int64

## 12. 날짜 컬럼 변환하기

날짜처럼 보이는 값도 실제로는 문자열일 수 있습니다. 월별 주문 분석이나 요일별 분석을 하려면 날짜형으로 변환해야 합니다.

`errors="coerce"`는 변환할 수 없는 값을 `NaT`로 바꿉니다. 그래서 변환 후 실패 건수를 반드시 확인해야 합니다.


In [15]:
orders_clean['order_date'] = pd.to_datetime(orders_clean['order_date'], errors='coerce')
print('order_date 변환 실패:', orders_clean['order_date'].isna().sum())
print('주문 시작일:', orders_clean['order_date'].min())
print('주문 종료일:', orders_clean['order_date'].max())

orders_clean['order_month'] = orders_clean['order_date'].dt.to_period('M').astype(str)
orders_clean['order_dayofweek'] = orders_clean['order_date'].dt.day_name()

orders_clean[['order_date', 'order_month', 'order_dayofweek']].head()


order_date 변환 실패: 3
주문 시작일: 2025-07-01 00:00:00
주문 종료일: 2028-12-31 00:00:00


,order_date,order_month,order_dayofweek
0,2026-07-12,2026-07,Sunday
1,2025-10-02,2025-10,Thursday
2,2026-03-13,2026-03,Friday
3,2025-10-12,2025-10,Sunday
4,2026-04-17,2026-04,Friday


In [16]:
if 'signup_date' in customers_clean.columns:
    customers_clean['signup_date'] = pd.to_datetime(customers_clean['signup_date'], errors='coerce')
    print('signup_date 변환 실패:', customers_clean['signup_date'].isna().sum())
    display(customers_clean[['customer_id', 'signup_date']].head())


signup_date 변환 실패: 2


,customer_id,signup_date
0,1,2025-10-05
1,2,2023-09-20
2,3,2025-01-22
3,4,2025-03-13
4,5,2024-02-10


## 13. 숫자형 컬럼 변환하기

가격, 수량, 단가는 숫자처럼 보여도 문자열일 수 있습니다. 특히 `10,000`처럼 쉼표가 들어간 값은 바로 계산하기 어렵습니다.


In [17]:
def to_number(series):
    return pd.to_numeric(
        series.astype(str).str.replace(',', '', regex=False),
        errors='coerce'
    )

products_clean['price'] = to_number(products_clean['price'])
order_items_clean['quantity'] = to_number(order_items_clean['quantity'])
order_items_clean['unit_price'] = to_number(order_items_clean['unit_price'])

print('price 변환 실패:', products_clean['price'].isna().sum())
print('quantity 변환 실패:', order_items_clean['quantity'].isna().sum())
print('unit_price 변환 실패:', order_items_clean['unit_price'].isna().sum())


price 변환 실패: 2
quantity 변환 실패: 5
unit_price 변환 실패: 2


## 과제 2. 문자열·타입·결측 처리

- 적용한 문자열 정리: object 컬럼 전체에 앞뒤 공백 제거. city의 ' 서울 ', payment_method의 ' card ', category의 '전자기기 ' 같은 값이 해당한다
- 숫자 변환 대상과 실패 건수: price 2건 / quantity 5건 / unit_price 2건
- 날짜 변환 대상과 실패 건수: order_date 3건 / signup_date 2건
- 결측 처리 기준: age는 중앙값 41.0으로 대체, city는 'Unknown'으로 대체

![타입과 결측 처리](images/step02_cleaning.png)

### 결과 관찰

공백을 제거하자 city 고유값이 15종에서 10종으로 줄었다. 숫자 변환에 실패한 값은 price가 'unknown', unit_price가 'unknown'과 'not-a-number', quantity가 '3개', '2개', '5개', 'three', 'unknown'이었다. 날짜 변환 실패는 'not-a-date', '2026-02-30', '2026/99/99', '2026-13-40'이다. age는 변환 전 결측이 4건이었는데 'unknown' 2건이 더해져 6건이 됐고 전부 41.0으로 채워졌다.

### 나의 해석과 판단

age를 중앙값으로 채운 건 6건이 152건 중 4% 정도라 분포를 크게 흔들지 않을 것 같아서다. city는 지역이라 중앙값 같은 게 없으니 'Unknown'을 따로 두는 쪽이 낫다고 봤다.

정보 손실이 걱정되는 건 quantity다. 변환 실패 5건 중 '3개', '2개', '5개' 3건은 단위만 붙은 거라 '개'만 떼면 그대로 쓸 수 있는 값이다. 그런데 to_number가 이걸 NaN으로 만들고, 뒤에서 quantity > 0 조건을 거는 순간 행 자체가 사라진다. 살릴 수 있는 주문 3건을 버리는 셈이다. 'three'와 'unknown'은 숫자를 추정할 근거가 없으니 남겨둘 이유가 없지만, '3개'는 다르다고 생각한다.

날짜의 '2026-02-30'도 비슷하다. 2월 30일은 없는 날이지만 입력 실수로 보이니 원본을 확인하면 복구될 가능성이 있다. 반면 'not-a-date'는 복구할 게 없다.

### 업무·분석적 의미

변환 실패를 전부 같은 방식으로 버리면 데이터 품질 문제가 아니라 수집 양식 문제인 건까지 묶여서 사라진다. '3개'처럼 단위가 붙는 패턴이 반복되면 입력 양식을 고쳐야 하는 문제고, 그건 데이터를 지운다고 해결되지 않는다.

### 한계와 추가 확인 사항

이번에는 노트북이 준 기준을 그대로 따랐기 때문에 '3개'를 복구하지 않았다. 실제로 쓸 거라면 단위를 떼는 규칙을 먼저 넣고 그 다음에 남는 실패만 버리는 순서가 맞다고 본다. age 중앙값 대체도 결측인 고객과 41세 고객이 섞여버려서 나중에 나이대별로 볼 때 41 근처가 부풀 수 있다.

## 14. 이상값 후보 확인하기

이상값은 일반적인 범위를 벗어난 값입니다. 하지만 이상값이 항상 오류는 아닙니다. 0원 상품은 이벤트 상품일 수 있고, 음수 수량은 반품을 의미할 수도 있습니다. 먼저 확인하고 기준을 정해야 합니다.


In [18]:
display(products_clean['price'].describe())
display(order_items_clean[['quantity', 'unit_price']].describe())


count    9.800000e+01
mean     2.005102e+05
std      1.001829e+06
min     -2.000000e+04
25%      5.200000e+04
50%      9.250000e+04
75%      1.527500e+05
max      9.999999e+06
Name: price, dtype: float64

,quantity,unit_price
count,758.000000,761.000000
mean,3.168865,103494.086728
std,4.159055,56969.823769
min,-1.000000,0.000000
25%,2.000000,55000.000000
50%,3.000000,96000.000000
75%,4.000000,154000.000000
max,100.000000,197000.000000


In [19]:
print('price <= 0:', len(products_clean[products_clean['price'] <= 0]))
print('quantity <= 0:', len(order_items_clean[order_items_clean['quantity'] <= 0]))
print('unit_price <= 0:', len(order_items_clean[order_items_clean['unit_price'] <= 0]))


price <= 0: 2
quantity <= 0: 3
unit_price <= 0: 1


### 실습용 이상값 처리 기준

이번 실습에서는 정상 주문 분석을 위해 다음 기준을 적용합니다.

```text
- price가 0 이하인 상품은 분석 대상에서 제외
- quantity가 0 이하인 주문 상세는 분석 대상에서 제외
- unit_price가 0 이하인 주문 상세는 분석 대상에서 제외
```

실제 업무에서는 이 기준을 적용하기 전에 반드시 원본 시스템이나 담당자 확인이 필요합니다.


In [20]:
products_clean = products_clean[products_clean['price'] > 0]
order_items_clean = order_items_clean[order_items_clean['quantity'] > 0]
order_items_clean = order_items_clean[order_items_clean['unit_price'] > 0]

print('이상값 처리 후 products:', products_clean.shape)
print('이상값 처리 후 order_items:', order_items_clean.shape)


이상값 처리 후 products: (96, 4)
이상값 처리 후 order_items: (752, 5)


## 과제 3. 중복·범주 표준화·이상값 후보

- 제거/보류한 중복: 네 파일에서 전체 행 중복 2건씩 제거. 보류한 건 없다
- 표준화한 범주값: order_status 15종을 completed / cancelled / refunded 3종으로 통일
- 허용값 밖 값: price 0 이하 2건, quantity 0 이하 3건, unit_price 0 이하 1건
- 이상값 후보: price 9999999, quantity 100과 50, customers의 age 150, 주문일 2027-01-15와 2028-12-31

![중복 범주 이상값](images/step03_rules.png)

### 결과 관찰

order_status는 completed / cancelled / refunded 외에 Complete, COMPLETED, 완료, cancel, Cancel, CANCELLED, 취소, refund, Refund, REFUNDED, 환불이 섞여 있었다. 매핑 후 completed 180 / cancelled 63 / refunded 57이 됐다. price는 평균 200510에 표준편차가 1001829으로 평균보다 훨씬 크다. 최댓값 9999999 하나가 끌어올린 값이다. quantity도 최대 100이 있다. 이상값 처리 후 products는 96행, order_items는 752행이 됐다.

### 나의 해석과 판단

이상값을 바로 지우면 안 되는 이유는 지금 이상값 후보로 잡힌 것들의 성격이 서로 다르기 때문이다. price -20000은 값 자체가 불가능하니 오류가 맞다. 그런데 quantity 100은 숫자로는 가능한 값이다. 도매 주문이면 실제로 나올 수 있고, 그런 주문을 지우면 매출 상위 건을 통째로 날리게 된다. 9999999도 입력 실수일 가능성이 높지만 확인 전에는 단정할 수 없다.

지우는 작업은 되돌릴 수 없으니, 후보로 표시해두고 왜 그런 값이 들어왔는지 확인한 다음에 판단하는 쪽이 안전하다고 본다.

노트북 기준으로는 잡히지 않은 것도 있다. age 150은 사람 나이로 불가능한데 customers에는 이상값 필터를 걸지 않아서 그대로 남았다. 주문일 2028-12-31도 날짜 형식은 멀쩡해서 변환에 성공했지만 오늘 기준으로 2년 뒤 주문이라 실제로는 있을 수 없다. 형식이 맞다고 값이 맞는 건 아니다.

### 업무·분석적 의미

범주값이 15종으로 흩어져 있으면 주문 상태별 집계가 그대로 틀린다. completed로 세야 할 건이 Complete, COMPLETED, 완료로 쪼개져서 실제 완료 건수가 175건으로 나오는데 맞는 값은 180건이다. 표준화 전후로 5건 차이가 난다.

### 한계와 추가 확인 사항

status_map에 적힌 표기만 바꾸므로 여기 없는 새 표기가 들어오면 그대로 남는다. 매핑 후에 값 종류를 다시 세는 검증이 필요하다. 그리고 age와 주문일에는 범위 검증을 걸지 않아서 150세와 미래 날짜가 남아 있다.

## 15. 파생 컬럼 만들기

전처리된 수량과 단가를 사용하면 주문 상세 금액을 계산할 수 있습니다.

`line_total = quantity × unit_price`


In [21]:
order_items_clean['line_total'] = (
    order_items_clean['quantity'] * order_items_clean['unit_price']
)

order_items_clean[['quantity', 'unit_price', 'line_total']].head()


,quantity,unit_price,line_total
0,2.0,172000.0,344000.0
1,4.0,79000.0,316000.0
2,3.0,87000.0,261000.0
3,1.0,187000.0,187000.0
4,5.0,125000.0,625000.0


In [22]:
print('전처리 후 주문 상세 금액 합계:', order_items_clean['line_total'].sum())


전처리 후 주문 상세 금액 합계: 252140000.0


## 16. 파일 간 관계 다시 확인하기

전처리 과정에서 일부 행을 삭제하면 파일 간 관계가 깨질 수 있습니다. 예를 들어 `products`에서 일부 상품을 제외했는데 `order_items`에는 그 상품 ID가 남아 있을 수 있습니다.


In [23]:
invalid_customers = orders_clean[
    ~orders_clean['customer_id'].isin(customers_clean['customer_id'])
]

invalid_orders = order_items_clean[
    ~order_items_clean['order_id'].isin(orders_clean['order_id'])
]

invalid_products = order_items_clean[
    ~order_items_clean['product_id'].isin(products_clean['product_id'])
]

relationship_checks = pd.DataFrame({
    'check': [
        'orders.customer_id exists in customers.customer_id',
        'order_items.order_id exists in orders.order_id',
        'order_items.product_id exists in products.product_id',
    ],
    'invalid_count': [
        len(invalid_customers),
        len(invalid_orders),
        len(invalid_products),
    ],
})

relationship_checks


,check,invalid_count
0,orders.customer_id exists in customers.custome...,2
1,order_items.order_id exists in orders.order_id,2
2,order_items.product_id exists in products.prod...,26


## 과제 4. PK/FK와 파생 컬럼 재검증

- 없는 `customer_id`: 2건
- 없는 `order_id`: 2건
- 없는 `product_id`: 26건
- 생성한 파생 컬럼: order_month, order_dayofweek, line_total
- `line_total` 검증: quantity × unit_price로 계산했고 합계는 252140000이다

![관계와 파생 컬럼 검증](images/step04_validation.png)

### 결과 관찰

product_id 미존재가 26건으로 다른 둘보다 훨씬 많다. 전처리 전 products를 기준으로 같은 검사를 돌리면 2건이다. 전처리 과정에서 24건이 새로 생긴 것이다. 이상값 처리로 products에서 빠진 product_id는 18, 48, 73, 95 네 개이고, 이 네 상품을 참조하는 order_items가 정확히 24행이다.

### 나의 해석과 판단

전처리 후에 관계를 다시 봐야 하는 이유가 여기서 그대로 나왔다. 각 파일을 따로 정리하면 파일 안에서는 깨끗해지지만 파일 사이의 연결은 그때 끊어진다. products에서 price가 이상한 4행을 지운 건 products만 보면 맞는 처리인데, 그 상품이 팔린 기록 24건이 상품 정보 없는 주문이 됐다.

이 상태로 상품별 매출을 집계하면 24건이 조용히 빠진다. merge를 inner로 걸면 결과에서 사라지고 행 수를 세보기 전에는 알아채기 어렵다. 전처리 전에 통과하던 검사가 전처리 후에 깨질 수 있으니 같은 검사를 양쪽에서 돌려야 한다.

## 17. 전처리 전후 비교하기

전처리 전후 데이터 크기를 비교합니다. 행 수가 줄었다면 어떤 기준으로 줄었는지 설명할 수 있어야 하고, 열 수가 늘었다면 어떤 파생 컬럼이 추가되었는지 설명할 수 있어야 합니다.


In [24]:
processed_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

processed_shapes = pd.DataFrame({
    'dataset': list(processed_data.keys()),
    'rows': [df.shape[0] for df in processed_data.values()],
    'columns': [df.shape[1] for df in processed_data.values()],
})

comparison = raw_shapes.merge(
    processed_shapes,
    on='dataset',
    suffixes=('_raw', '_processed')
)

comparison


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,152,6,150,6
1,products,102,4,96,4
2,orders,302,5,300,7
3,order_items,765,5,752,6


## 과제 5. 전처리 전/후 비교

| 항목 | 처리 전 | 처리 후 | 변화 이유 |
| --- | ---: | ---: | --- |
| 행 수 | 152 / 102 / 302 / 765 | 150 / 96 / 300 / 752 | 완전 중복 2건씩 제거, products는 price 이상/결측 4행, order_items는 quantity와 unit_price 이상/결측 11행 추가 제거 |
| 결측 | customers age 4, city 2 | age 0, city 0 / signup_date 2, order_date 3, order_month 3 | age는 중앙값 41.0, city는 Unknown으로 대체. 날짜 변환 실패분은 NaT로 남아 결측이 새로 생겼다 |
| 중복 | 네 파일 각 2건 | 0 | drop_duplicates로 완전 중복 행 제거 |
| 변환 실패 | price 2, quantity 5, unit_price 2, order_date 3, signup_date 2 | 날짜 5건은 NaT로 남음 | 숫자 변환 실패는 0 이하 필터에서 행째 제거, 날짜는 NaT 상태로 유지 |

![전처리 전후 비교](images/step05_before_after.png)

### 결과 관찰

컬럼 수도 늘었다. orders는 5개에서 7개로, order_items는 5개에서 6개로 바뀌었다. 파생 컬럼을 만들었기 때문이다. 가장 많이 줄어든 건 products로 102행에서 96행이 됐다.

결측은 0이 되지 않았다. customers의 age와 city는 채웠지만 signup_date 2건이 NaT로 남았고, orders는 order_date와 거기서 만든 order_month, order_dayofweek이 각각 3건씩 결측이다. 처리 전에는 isna로 세면 0이던 컬럼들이다.

### 나의 해석과 판단

행이 줄어든 건 대체로 의도한 결과지만 products 쪽은 그렇지 않다. 중복 2건을 빼면 100행인데 96행이 됐으니 4행이 더 빠졌다. price가 0 이하인 2건은 의도한 제거가 맞다. 나머지 2건은 price가 'unknown'이라 숫자 변환에 실패한 행인데, price > 0 조건에서 NaN은 False가 되니까 같이 빠졌다.

변환 실패와 범위 위반은 원인이 다른데 한 줄의 필터가 둘을 똑같이 처리했다. 그래서 숫자 변환 실패 건수는 출력에 찍혀도 그 행들이 어디로 갔는지는 어디에도 안 남는다.

날짜 변환 실패 5건은 다르다. NaT로 남아서 행이 유지된다. 같은 변환 실패인데 숫자는 행이 사라지고 날짜는 남는 것이다.

### 업무·분석적 의미

행 수가 줄어든 이유를 항목별로 적어두지 않으면 나중에 매출 합계가 맞지 않을 때 어디서 빠졌는지 찾을 수 없다. 전처리 전후 행 수 비교를 보고서에 남겨야 하는 이유다.

### 한계와 추가 확인 사항

지금 비교표는 행 수와 결측, 중복만 본다. 값 분포가 어떻게 바뀌었는지는 안 본다. age를 중앙값으로 채운 뒤 41 근처 빈도가 얼마나 늘었는지 같은 건 따로 확인해야 한다.

## 18. 전처리 결과 저장하기

전처리된 데이터는 `data/processed` 폴더에 저장합니다. Excel에서 한글 CSV를 바로 열 수 있도록 `utf-8-sig` 인코딩을 사용합니다.


In [25]:
customers_clean.to_csv(PROCESSED_DIR / 'customers_clean.csv', index=False, encoding='utf-8-sig')
products_clean.to_csv(PROCESSED_DIR / 'products_clean.csv', index=False, encoding='utf-8-sig')
orders_clean.to_csv(PROCESSED_DIR / 'orders_clean.csv', index=False, encoding='utf-8-sig')
order_items_clean.to_csv(PROCESSED_DIR / 'order_items_clean.csv', index=False, encoding='utf-8-sig')

list(PROCESSED_DIR.glob('*_clean.csv'))


[WindowsPath('C:/Users/yjun0/Desktop/서울대/4학년/2학기/LLM을 이용한 빅데이터 분석/llm-data-analysis-course/data/processed/customers_clean.csv'),
 WindowsPath('C:/Users/yjun0/Desktop/서울대/4학년/2학기/LLM을 이용한 빅데이터 분석/llm-data-analysis-course/data/processed/orders_clean.csv'),
 WindowsPath('C:/Users/yjun0/Desktop/서울대/4학년/2학기/LLM을 이용한 빅데이터 분석/llm-data-analysis-course/data/processed/order_items_clean.csv'),
 WindowsPath('C:/Users/yjun0/Desktop/서울대/4학년/2학기/LLM을 이용한 빅데이터 분석/llm-data-analysis-course/data/processed/products_clean.csv')]

## 19. 전처리 요약 보고서 저장하기

전처리 기준과 결과를 Markdown 파일로 남겨 두면 이후 보고서 작성이 쉬워집니다.


In [26]:
summary_text = f'''# Chapter 5 데이터 전처리 요약

## 전처리 결과 파일

- customers_clean.csv
- products_clean.csv
- orders_clean.csv
- order_items_clean.csv

## 전처리 전후 데이터 크기

```text
{comparison.to_string(index=False)}
```

## 파일 간 관계 점검 결과

```text
{relationship_checks.to_string(index=False)}
```

## 주요 처리 내용

- 원본 데이터를 직접 수정하지 않고 복사본을 사용함
- 문자열 컬럼 앞뒤 공백 제거
- 고객 나이 결측치는 중앙값으로 대체
- 고객 도시 결측치는 Unknown으로 처리
- 주문 상태값 표기 통일
- 날짜 컬럼 변환 및 주문 월/요일 파생 컬럼 생성
- 가격, 수량, 단가를 숫자형으로 변환
- 0 이하 가격, 수량, 단가 확인 및 처리
- line_total 파생 컬럼 생성
- 파일 간 키 관계 재확인

## 주의 사항

이 전처리 기준은 실습용 예시입니다. 실제 업무에서는 결측치와 이상값을 삭제하거나 대체하기 전에 원본 시스템, 수집 과정, 업무 담당자 확인이 필요합니다.
'''

report_path = REPORT_DIR / 'ch05_preprocessing_summary.md'
report_path.write_text(summary_text, encoding='utf-8')

print('요약 보고서 저장 완료:', report_path)


요약 보고서 저장 완료: C:\Users\yjun0\Desktop\서울대\4학년\2학기\LLM을 이용한 빅데이터 분석\llm-data-analysis-course\reports\ch05_preprocessing_summary.md


## 20. 전처리 과정을 함수로 정리하기

전처리 코드를 한 번만 실행하고 끝내면 재사용하기 어렵습니다. 같은 데이터를 다시 받거나 처리 기준을 조금 바꾸어야 할 때는 함수로 정리된 코드가 훨씬 유용합니다.


In [27]:
def preprocess_customers(df):
    result = strip_string_columns(df)

    if 'age' in result.columns:
        result['age'] = pd.to_numeric(result['age'], errors='coerce')
        result['age'] = result['age'].fillna(result['age'].median())

    if 'city' in result.columns:
        result['city'] = result['city'].fillna('Unknown')

    if 'signup_date' in result.columns:
        result['signup_date'] = pd.to_datetime(result['signup_date'], errors='coerce')

    return result.drop_duplicates()


def preprocess_products(df):
    result = strip_string_columns(df)

    if 'price' in result.columns:
        result['price'] = to_number(result['price'])
        result = result[result['price'] > 0]

    return result.drop_duplicates()


def preprocess_orders(df):
    result = strip_string_columns(df)

    if 'order_status' in result.columns:
        result['order_status'] = result['order_status'].replace(status_map)

    if 'order_date' in result.columns:
        result['order_date'] = pd.to_datetime(result['order_date'], errors='coerce')
        result['order_month'] = result['order_date'].dt.to_period('M').astype(str)
        result['order_dayofweek'] = result['order_date'].dt.day_name()

    return result.drop_duplicates()


def preprocess_order_items(df):
    result = strip_string_columns(df)

    if 'quantity' in result.columns:
        result['quantity'] = to_number(result['quantity'])
        result = result[result['quantity'] > 0]

    if 'unit_price' in result.columns:
        result['unit_price'] = to_number(result['unit_price'])
        result = result[result['unit_price'] > 0]

    if {'quantity', 'unit_price'}.issubset(result.columns):
        result['line_total'] = result['quantity'] * result['unit_price']

    return result.drop_duplicates()


In [28]:
customers_clean_func = preprocess_customers(customers)
products_clean_func = preprocess_products(products)
orders_clean_func = preprocess_orders(orders)
order_items_clean_func = preprocess_order_items(order_items)

print(customers_clean_func.shape)
print(products_clean_func.shape)
print(orders_clean_func.shape)
print(order_items_clean_func.shape)


(150, 6)
(96, 4)
(300, 7)
(752, 6)


C:\Users\yjun0\AppData\Local\Temp\ipykernel_15244\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = result.select_dtypes(include='object').columns
C:\Users\yjun0\AppData\Local\Temp\ipykernel_15244\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/us

## 21. 소스 모듈 사용하기

위에서 직접 작성한 전처리 함수는 `src/preprocessing.py`에도 정리되어 있습니다. 반복 작업이나 실제 프로젝트에서는 노트북 안에만 코드를 두기보다 소스 모듈로 분리하는 것이 좋습니다.


In [29]:
from src.data_loader import load_sales_data
from src.preprocessing import (
    build_preprocessing_report,
    compare_shapes,
    duplicate_summary,
    preprocess_sales_data,
    save_processed_data,
    validate_relationships,
)

module_raw_data = load_sales_data(RAW_DIR)
module_processed_data = preprocess_sales_data(module_raw_data)
module_relationship_checks = validate_relationships(module_processed_data)
module_comparison = compare_shapes(module_raw_data, module_processed_data)

module_comparison


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,152,6,150,6
1,order_items,765,5,752,6
2,orders,302,5,300,7
3,products,102,4,96,4


In [30]:
duplicate_summary(
    module_processed_data,
    key_columns={
        'customers': 'customer_id',
        'products': 'product_id',
        'orders': 'order_id',
        'order_items': 'order_item_id',
    },
)


,dataset,row_duplicate_count,key_column,key_duplicate_count
0,customers,0,customer_id,0
1,products,0,product_id,0
2,orders,0,order_id,0
3,order_items,0,order_item_id,0


In [31]:
module_relationship_checks


,check,invalid_count
0,orders.customer_id exists in customers.custome...,2
1,order_items.order_id exists in orders.order_id,2
2,order_items.product_id exists in products.prod...,26


## 22. 스크립트로 한 번에 실행하기

노트북에서 한 단계씩 이해한 전처리 과정을 스크립트로도 실행할 수 있습니다. 터미널에서 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/preprocess_data.py
```

이 스크립트는 전처리 결과 CSV와 요약 보고서를 자동으로 저장합니다.


## 과제 6. 재현 가능한 전처리 확인

- 생성된 clean CSV: data/processed에 customers_clean.csv, products_clean.csv, orders_clean.csv, order_items_clean.csv
- 생성된 요약 보고서: reports/ch05_preprocessing_summary.md
- `python scripts/preprocess_data.py` 재실행 결과: 문서에 적힌 그대로 실행하면 ModuleNotFoundError: No module named 'src'가 난다. PYTHONPATH에 프로젝트 루트를 넣고 실행하면 정상 동작한다
- 재실행 후 달라진 점: clean CSV 4개는 노트북이 저장한 것과 바이트 단위로 같다. reports/ch05_preprocessing_summary.md만 다른데, 스크립트판에는 중복 점검 결과 섹션이 더 들어가고 데이터셋 정렬 순서와 문장 표현이 다르다

![재실행 결과](images/step06_reproducible.png)

### 나의 해석과 판단

스크립트를 실행하려는데 바로 실패했다. scripts/preprocess_data.py는 from src.data_loader import load_sales_data로 시작하는데, python scripts/preprocess_data.py로 실행하면 sys.path 맨 앞이 scripts 폴더라 프로젝트 루트의 src를 못 찾는다. 노트북에서는 셀 5가 sys.path에 루트를 직접 넣어줘서 같은 import가 통과했다. 노트북에서 되던 게 스크립트에서 안 되는 이유가 코드가 아니라 실행 위치에 있었다.

노트북에서 한 번 성공하는 것과 스크립트로 재현되는 것의 차이가 이 지점이라고 생각한다. 노트북은 셀을 순서대로 돌리는 동안 변수와 경로가 메모리에 남아 있어서, 앞 셀이 만들어둔 상태에 기대는 코드도 그냥 돌아간다. 스크립트는 매번 빈 상태에서 시작하니까 그런 전제가 전부 드러난다. 노트북에서 돌아갔다는 건 그 순서로 한 번 돌아갔다는 뜻이지 다음에도 같은 결과가 나온다는 보장은 아니다.

처음 비교했을 때는 clean CSV도 하나 달랐다. orders_clean.csv에서 날짜 변환에 실패한 3행의 order_month가 노트북에서는 'NaT' 문자열로, 스크립트에서는 빈 값으로 저장됐다. 코드는 양쪽 다 to_period('M').astype(str)로 같은데 결과가 달랐다.

원인은 노트북을 돌린 커널이 프로젝트 .venv가 아니라 anaconda 쪽 python이었다는 것이다. 커널의 pandas는 2.2.2였고 .venv는 3.0.5라서 NaT를 문자열로 만드느냐 결측으로 두느냐가 갈렸다. 커널을 .venv로 바꾸고 다시 실행하니 clean CSV 4개가 전부 같아졌다. Chapter 02에서 터미널 Python과 Jupyter 커널을 맞춰야 한다고 한 이유를 여기서 다시 확인했다.

노트북은 셀을 위에서부터 실행하지 않으면 결과가 달라진다는 점도 있다. 중복 제거와 범주 표준화가 서로 다른 셀에 있어서 순서를 바꾸면 남는 행이 달라질 수 있다. 스크립트는 그 순서가 함수 안에 고정돼 있다.

## 23. LLM과 함께 전처리 코드를 검토하기

LLM에게 전처리 코드를 요청할 때는 실제 고객명, 이메일, 주문 내역을 그대로 입력하지 않는 것이 좋습니다. 컬럼명, 데이터 타입, 결측치 개수, 중복 여부, 처리 목적처럼 구조화된 정보만 제공하세요.

```text
다음 전처리 코드가 안전한지 검토해 주세요.

customers["age"] = customers["age"].fillna(customers["age"].mean())
customers = customers.drop_duplicates()
orders["order_date"] = pd.to_datetime(orders["order_date"])
order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]

검토 기준:
- 원본 데이터를 직접 수정하는 문제가 있는지
- 결측치 처리 방식이 적절한지
- 날짜 변환 실패를 확인하는지
- 중복 제거 기준이 충분한지
- line_total 계산 전에 숫자형 변환이 필요한지
- 더 안전한 코드로 어떻게 수정할 수 있는지
```


## 24. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 각 데이터셋의 결측치 비율을 하나의 표로 합쳐 보세요.
2. `customers_clean`에서 나이가 18세 미만이거나 100세 초과인 값이 있는지 확인하세요.
3. `orders_clean`에서 월별 주문 건수를 계산하세요.
4. `order_items_clean`에서 `line_total`이 큰 순서대로 상위 10개를 확인하세요.
5. 전처리 전후 행 수가 줄어든 데이터셋이 있는지 설명해 보세요.
6. LLM에게 결측치 처리 기준을 검토해 달라는 프롬프트를 직접 작성해 보세요.


In [32]:
# 과제 1. 각 데이터셋의 결측치 비율을 하나의 표로 합쳐 보세요.
# hint: missing_summary(df).reset_index()를 활용해 보세요.

missing_all = []

for name, df in processed_data.items():
    summary = missing_summary(df).reset_index()
    summary = summary.rename(columns={'index': 'column'})
    summary.insert(0, 'dataset', name)
    missing_all.append(summary)

missing_all = pd.concat(missing_all, ignore_index=True)
missing_all

,dataset,column,missing_count,missing_ratio
0,customers,signup_date,2,1.33
1,customers,customer_id,0,0.00
2,customers,name,0,0.00
3,customers,gender,0,0.00
4,customers,age,0,0.00
5,customers,city,0,0.00
6,products,product_id,0,0.00
7,products,product_name,0,0.00
8,products,category,0,0.00
9,products,price,0,0.00


In [33]:
# 과제 2. customers_clean에서 나이가 18세 미만이거나 100세 초과인 값이 있는지 확인하세요.

age_out = customers_clean[
    (customers_clean['age'] < 18) | (customers_clean['age'] > 100)
]

print('범위 밖 나이:', len(age_out))
age_out[['customer_id', 'name', 'age']]

범위 밖 나이: 1


,customer_id,name,age
32,33,권승윤,150.0


In [34]:
# 과제 3. orders_clean에서 월별 주문 건수를 계산하세요.

monthly_orders = orders_clean['order_month'].value_counts().sort_index()
monthly_orders

order_month
2025-07    19
2025-08    26
2025-09    12
2025-10    22
2025-11    23
2025-12    23
2026-01    26
2026-02    19
2026-03    22
2026-04    26
2026-05    20
2026-06    24
2026-07    12
2026-08    21
2027-01     1
2028-12     1
Name: count, dtype: int64

In [35]:
# 과제 4. order_items_clean에서 line_total이 큰 순서대로 상위 10개를 확인하세요.

order_items_clean.sort_values('line_total', ascending=False).head(10)

,order_item_id,order_id,product_id,quantity,unit_price,line_total
750,751,295,31,100.0,155000.0,15500000.0
737,738,290,96,50.0,96000.0,4800000.0
319,320,129,23,5.0,197000.0,985000.0
9,10,5,65,5.0,193000.0,965000.0
24,25,11,65,5.0,193000.0,965000.0
293,294,119,36,5.0,193000.0,965000.0
116,117,51,65,5.0,193000.0,965000.0
20,21,9,36,5.0,193000.0,965000.0
290,291,117,47,5.0,188000.0,940000.0
491,492,195,99,5.0,187000.0,935000.0


## 25. 정리

이번 장에서는 다음 내용을 실습했습니다.

- 원본 데이터와 전처리 데이터 분리
- 결측치 개수와 비율 확인
- 나이 결측치 중앙값 대체, 도시 결측치 Unknown 처리
- 중복 행과 주요 ID 중복 점검
- 문자열 앞뒤 공백 제거와 상태값 표기 통일
- 날짜형 변환과 주문 월/요일 파생 컬럼 생성
- 숫자형 변환과 변환 실패 건수 확인
- 이상값 후보 확인과 실습용 처리 기준 적용
- 주문 상세 금액 `line_total` 생성
- 파일 간 키 관계 재확인
- 전처리 결과 CSV와 요약 보고서 저장
- 전처리 함수를 `src/preprocessing.py` 소스 모듈로 분리

다음 장에서는 전처리된 데이터를 바탕으로 탐색적 데이터 분석, 즉 EDA를 수행합니다.


## 과제 7. Chapter 05 최종 판단

### 내가 정한 전처리 원칙 3가지

1. 원본은 건드리지 않고 복사본에서만 처리한다. 이번에도 raw 폴더는 읽기만 했고 결과는 data/processed에 따로 저장했다.
2. 행을 지우는 조건은 지우려는 이유 하나만 보게 만든다. price > 0 같은 조건은 범위 위반과 변환 실패를 같이 지우기 때문에, 변환 실패는 따로 세고 따로 처리한다.
3. 파일 안을 정리한 뒤에는 파일 사이 관계를 다시 검사한다. 전처리 전에 통과하던 검사가 전처리 후에 깨질 수 있다.

### 가장 위험하다고 생각한 자동 처리 1가지와 이유

products_clean = products_clean[products_clean['price'] > 0] 한 줄이다.

이 줄은 price가 0 이하인 행을 지우려고 쓴 건데, NaN과 비교하면 False가 나오기 때문에 숫자 변환에 실패한 행도 같이 지운다. products는 중복 제거 후 100행이었고 이 줄을 지나면 96행이 된다. 빠진 4행 중 2행은 price가 0 이하였고 나머지 2행은 price가 'unknown'이었다.

위험한 건 지워진다는 사실보다 아무 표시가 안 남는다는 점이다. 바로 위 셀에서 price 변환 실패 2건을 출력까지 했는데, 그 2건이 이 줄에서 사라진 건 출력 어디에도 안 나온다. 행 수만 보면 4행이 줄었다는 것만 보이고 그 안에 성격이 다른 두 종류가 섞여 있다는 건 따로 세보지 않으면 모른다.

그리고 이 4행이 빠지면서 order_items의 product_id 미존재가 2건에서 26건으로 늘었다. 한 줄의 필터가 다른 파일의 관계까지 깼다.

### 다음 EDA에서 특히 주의할 데이터 특성

product_id 미존재 26건을 먼저 봐야 한다. 상품별 매출을 inner join으로 집계하면 이 26건이 말없이 빠지므로, merge 전후 행 수를 세고 어느 상품이 빠졌는지 확인해야 한다.

날짜도 그대로 쓰면 안 된다. order_date가 NaT인 행이 3건 남아 있고 order_month에 NaT 문자열이 들어간다. 주문일 최댓값이 2028-12-31이라 월별 추이를 그리면 2년 뒤까지 축이 늘어난다.

age는 중앙값으로 채운 6건이 41에 몰려 있고 150세가 그대로 남아 있다. 나이대별 분석을 할 거면 이 둘을 먼저 정리해야 한다.

### 현재 전처리의 한계

처리 기준을 노트북이 준 대로 따랐고 근거를 직접 확인하지는 않았다. quantity의 '3개'처럼 복구할 수 있는 값을 버렸고, age 150이나 미래 날짜처럼 형식은 맞지만 값이 불가능한 건은 아예 검사하지 않았다. 이상값 기준도 0 이하만 보고 상한은 안 봤다.

실제 업무라면 지우기 전에 원본 시스템에서 왜 그런 값이 들어왔는지 확인하는 단계가 있어야 한다.

## 최종 제출 체크

- [x] raw 원본을 덮어쓰지 않았습니다.
- [x] 처리 전/후 비교를 남겼습니다.
- [x] 처리 기준과 이유를 자신의 말로 작성했습니다.
- [x] clean CSV와 재실행 결과를 확인했습니다.
- [x] 개인정보/Secret이 없습니다.
- [x] `chapter05/chapter05.ipynb`가 GitHub에서 정상 표시됩니다.
- [x] 최종 Notebook 파일 URL을 제출합니다.